In [ ]:
!pip install --upgrade pip
!pip install matplotlib numpy termcolor scipy navpy
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


!git clone https://github.com/Preypatel2413/ai-imu-dr.git
%cd ai-imu-dr
!ls -la

Below is stage by stage preprocessing of data.

#stage 1 :

Stage 1 preprocessing for IMU/GPS pipeline.

This stage:
  - loads raw CSV exported from device,
  - keeps only the columns we care about,
  - optionally skips initial rows (to align initial timestamps),
  - renames timestamp columns to short names (GPST, AST, GST, MST, PST),
  - converts timestamps to a relative timebase by subtracting a rounded
    `time_offset` (ms),
  - writes the reduced CSV to disk.


In [ ]:
import numpy as np
import pandas as pd

input_file = "/content/ai-imu-dr/raw_data/input_data.csv"
output_file = "/content/ai-imu-dr/raw_data/data_stage_1.csv"

def stage1(input_path, output_path, prnt = False):

    INPUT_FILE = input_path
    OUTPUT_FILE = output_path

    df = pd.read_csv(INPUT_FILE, sep=",", engine= "python")

    cols_to_keep = [
        "GPS Time(ms)", "Lat", "Lon", "GPS Altitude", "GPS Speed", "Map Bearing",
        "Snapped Lat", "Snapped Lon",
        "Accel Sensor Time(ms)", "Ax", "Ay", "Az",
        "Gyro Sensor Time(ms)", "Gx", "Gy", "Gz",
        "Magn Sensor Time(ms)", "Mx", "My", "Mz",
        "Pressure Sensor Time(ms)", "Pressure"
    ]

    df = df[cols_to_keep]
    df = df[200:]                   # Skip initial rows to avoid mismatched initial timing bursts.

    df.rename(columns={
        'GPS Time(ms)': 'GPST',
        'Accel Sensor Time(ms)': 'AST',
        'Gyro Sensor Time(ms)': 'GST',
        'Magn Sensor Time(ms)': 'MST',
        'Pressure Sensor Time(ms)': 'PST'
    }, inplace=True)

    # All timestamps are epoch-based, so subtract the minimum (rounded to nearest second)
    mod = 1000
    time_offset = (min(df['GPST'].iloc[0], df['AST'].iloc[0], df['GST'].iloc[0], df['MST'].iloc[0], df['PST'].iloc[0])//mod )* mod

    print("Time offset : ", time_offset)
    df['GPST'] = df['GPST'] - time_offset
    df['AST'] = df['AST'] - time_offset
    df['GST'] = df['GST'] - time_offset
    df['MST'] = df['MST'] - time_offset
    df['PST'] = df['PST'] - time_offset

    df.to_csv(OUTPUT_FILE, index=False)
    if(prnt):
        print(f"Saved necessary columns -> {OUTPUT_FILE}")
        print(df.head(10))


stage1(input_file, output_file, True)

#stage 2


Stage 2 preprocessing for the IMU/GPS pipeline.

This stage:
  - Loads the cleaned CSV from stage1.
  - Chooses a master timebase unique(AST + GST).
  - Converts GPS lat/lon to local ENU coordinates.
  - Interpolates GPS position, altitude, speed, and bearing to IMU timestamps.
  - Computes ground-truth velocity (vx, vy, vz) and yaw from GPS bearing.
  - Synchronizes raw accel/gyro data to the master timeline using a ±window
    mean (handles sensor batching).
  - Builds a unified, time-aligned dataset containing:
        time (s), p_gt_*, v_gt_*, ang_gt_*, Ax/Ay/Az, Gx/Gy/Gz.
  - Detects large time gaps and fills them with interpolated samples.
  - Writes the synchronized and gap-filled CSV (data_stage_2).

Output is fully in sync IMU + GPS ground truth.

In [ ]:
import numpy as np
import pandas as pd

try:
    from pyproj import Transformer # type: ignore
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False

input_file = "/content/ai-imu-dr/raw_data/data_stage_1.csv"
output_file = "/content/ai-imu-dr/raw_data/data_stage_2.csv"

def stage2(input_path, output_path, prnt=False):
    INPUT = input_path
    OUTPUT = output_path

    # Which IMU timestamp to use as the master timebase
    PREFERRED_IMU_TIME = ["AST", "GST", "GPST"]  # accel -> gyro -> gps fallback
    USE_ENU = True
    ROLL_PITCH_NOISE_STD = 0.0
    MAX_GAP_S = 0.5

    # Window for accel/gyro synchronization (ms)
    WINDOW_HALF_MS = 15   # +-15 ms is safe and works perfectly on all Android phones

    # requested column groups / names
    gt_cols = ["p_gt_x","p_gt_y","p_gt_z","v_gt_x","v_gt_y","v_gt_z",
            "ang_gt_roll","ang_gt_pitch","ang_gt_yaw"]
    imu_cols = ["Ax","Ay","Az","Gx","Gy","Gz"]
    time_col = "time"

    df = pd.read_csv(INPUT)

    # Choose master timebase (prefers accel -> gyro -> gps)
    # imu_time = None
    # for cand in PREFERRED_IMU_TIME:
    #     if cand in df.columns:
    #         imu_time = cand
    #         break
    # if imu_time is None:
    #     raise ValueError(f"None of {PREFERRED_IMU_TIME} found. Columns: {df.columns.tolist()}")

    times = np.concatenate([
        df[col].dropna().values.astype(np.float64)
        for col in ['AST', 'GST']
    ])                                                  #this method uses unique timestamps from AST and GST

    t_target_ms = np.sort(np.unique(times))
    # t_target_ms = df[imu_time].values.astype(np.float64)   # master timestamps in ms

    # ============ GPS -> ENU + interpolation ============
    if "GPST" not in df.columns:
        df["GPST"] = df[imu_time]

    gps_df = df[["GPST", "Lat", "Lon", "GPS Altitude", "GPS Speed", "Map Bearing"]].copy()
    gps_df = gps_df.dropna(subset=["GPST"])
    gps_df = gps_df.groupby("GPST", as_index=False).first()     # when multiple rows have same GPST we keep the first (groupby.first)
    gps_df = gps_df.sort_values("GPST")

    t_gps = gps_df["GPST"].values.astype(np.float64)
    lat_gps = gps_df["Lat"].astype(float).values
    lon_gps = gps_df["Lon"].astype(float).values
    alt_gps = gps_df["GPS Altitude"].astype(float).values if "GPS Altitude" in gps_df.columns else np.full_like(t_gps, np.nan)
    speed_gps = gps_df["GPS Speed"].astype(float).values if "GPS Speed" in gps_df.columns else np.full_like(t_gps, np.nan)
    bearing_gps = gps_df["Map Bearing"].astype(float).values if "Map Bearing" in gps_df.columns else np.full_like(t_gps, np.nan)

    def latlon_to_enu(lats, lons, lat0=None, lon0=None):
        if lat0 is None: lat0 = float(np.mean(lats))
        if lon0 is None: lon0 = float(np.mean(lons))
        if HAVE_PYPROJ and USE_ENU:
            proj_str = f"+proj=aeqd +R=6378137 +lat_0={lat0} +lon_0={lon0}"
            transformer = Transformer.from_crs("epsg:4326", proj_str, always_xy=True)
            xs, ys = transformer.transform(lons.tolist(), lats.tolist())
            return np.array(xs), np.array(ys), lat0, lon0
        else:
            R = 6378137.0
            lat0_rad = np.deg2rad(lat0)
            dlat = np.deg2rad(lats - lat0)
            dlon = np.deg2rad(lons - lon0)
            x = R * dlon * np.cos(lat0_rad)
            y = R * dlat
            return x, y, lat0, lon0

    if len(lat_gps) == 0:
        raise ValueError("No GPS samples found!")

    x_gps, y_gps, lat0, lon0 = latlon_to_enu(lat_gps, lon_gps)

    order = np.argsort(t_gps)
    t_gps = t_gps[order]
    x_gps = x_gps[order]; y_gps = y_gps[order]
    alt_gps = alt_gps[order]; speed_gps = speed_gps[order]; bearing_gps = bearing_gps[order]

    def circular_interp_deg(t_src, angles_deg, t_tgt):
        ang_rad = np.deg2rad(np.array(angles_deg, dtype=float))
        sin_i = np.interp(t_tgt, t_src, np.sin(ang_rad), left=np.nan, right=np.nan)
        cos_i = np.interp(t_tgt, t_src, np.cos(ang_rad), left=np.nan, right=np.nan)
        ang_i = np.rad2deg(np.arctan2(sin_i, cos_i))
        return (ang_i + 360.0) % 360.0

    # Interpolate GPS fields
    x_target    = np.interp(t_target_ms, t_gps, x_gps, left=np.nan, right=np.nan)
    y_target    = np.interp(t_target_ms, t_gps, y_gps, left=np.nan, right=np.nan)
    alt_target  = np.interp(t_target_ms, t_gps, alt_gps, left=np.nan, right=np.nan)
    speed_target= np.interp(t_target_ms, t_gps, speed_gps, left=np.nan, right=np.nan)
    bearing_target = circular_interp_deg(t_gps, bearing_gps, t_target_ms)

    # Fill leading/trailing with nearest GPS
    if t_target_ms[0] < t_gps[0]:
        first = np.searchsorted(t_target_ms, t_gps[0])
        for arr, val in zip([x_target,y_target,alt_target,speed_target,bearing_target],
                            [x_gps[0],y_gps[0],alt_gps[0],speed_gps[0],bearing_gps[0]]):
            arr[:first] = val
    if t_target_ms[-1] > t_gps[-1]:
        last = np.searchsorted(t_target_ms, t_gps[-1], side="right")
        for arr, val in zip([x_target,y_target,alt_target,speed_target,bearing_target],
                            [x_gps[-1],y_gps[-1],alt_gps[-1],speed_gps[-1],bearing_gps[-1]]):
            arr[last:] = val

    # ============ VELOCITY FROM SPEED + BEARING ============
    bearing_rad = np.deg2rad(bearing_target)
    vx = speed_target * np.cos(bearing_rad)
    vy = speed_target * np.sin(bearing_rad)

    t_s = t_target_ms / 1000.0
    vz = np.zeros_like(alt_target)
    if len(t_s) >= 2:
        dt = np.diff(t_s)
        dalt = np.diff(alt_target)
        valid = dt > 1e-9
        vz_mid = np.zeros_like(dalt)
        vz_mid[valid] = dalt[valid] / dt[valid]
        vz[0] = vz_mid[0]
        vz[-1] = vz_mid[-1]
        if len(vz_mid) > 1:
            vz[1:-1] = 0.5 * (vz_mid[:-1] + vz_mid[1:])

    # ============ BUILD OUTPUT DATAFRAME ============
    df_out = pd.DataFrame()
    df_out[time_col] = t_s  # seconds

    df_out["p_gt_x"] = x_target
    df_out["p_gt_y"] = y_target
    df_out["p_gt_z"] = alt_target
    df_out["v_gt_x"] = vx
    df_out["v_gt_y"] = vy
    df_out["v_gt_z"] = vz

    yaw_rad = (np.deg2rad(bearing_target) + np.pi) % (2*np.pi) - np.pi       # yaw is derived from GPS bearing
    df_out["ang_gt_roll"]  = 0.0
    df_out["ang_gt_pitch"] = 0.0
    df_out["ang_gt_yaw"]   = yaw_rad

    if ROLL_PITCH_NOISE_STD > 0:
        rng = np.random.default_rng(42)
        df_out["ang_gt_roll"]  += rng.normal(0, ROLL_PITCH_NOISE_STD, len(df_out))
        df_out["ang_gt_pitch"] += rng.normal(0, ROLL_PITCH_NOISE_STD, len(df_out))


    # ============ ROBUST IMU SYNCHRONIZATION ============
    if(prnt):
        print(f"Synchronizing IMU using ±{WINDOW_HALF_MS} ms window averaging...")

    imu_parts = []
    if all(c in df.columns for c in ["AST", "Ax", "Ay", "Az"]):
        accel = df[["AST", "Ax", "Ay", "Az"]].rename(columns={"AST": "t_ms"})
        accel["sensor"] = "accel"
        imu_parts.append(accel)
    if all(c in df.columns for c in ["GST", "Gx", "Gy", "Gz"]):
        gyro = df[["GST", "Gx", "Gy", "Gz"]].rename(columns={"GST": "t_ms"})
        gyro["sensor"] = "gyro"
        imu_parts.append(gyro)

    if not imu_parts:
        raise ValueError("No accel or gyro data found!")

    imu_all = pd.concat(imu_parts, ignore_index=True)
    imu_all = imu_all.sort_values("t_ms").reset_index(drop=True)

    def window_mean_sync(t_native, values, t_target, half_win):
        out = np.full((len(t_target), values.shape[1]), np.nan)
        for i, t in enumerate(t_target):
            mask = (t_native >= t - half_win) & (t_native <= t + half_win)
            if mask.any():
                out[i] = np.mean(values[mask], axis=0)
            else:
                # fallback nearest
                idx = np.argmin(np.abs(t_native - t))
                out[i] = values[idx]
        return out

    # Extract raw data
    accel_mask = imu_all["sensor"] == "accel"
    gyro_mask  = imu_all["sensor"] == "gyro"

    accel_t = imu_all.loc[accel_mask, "t_ms"].values
    gyro_t  = imu_all.loc[gyro_mask,  "t_ms"].values

    accel_val = imu_all.loc[accel_mask, ["Ax","Ay","Az"]].values.astype(float)
    gyro_val  = imu_all.loc[gyro_mask,  ["Gx","Gy","Gz"]].values.astype(float)

    synced_accel = window_mean_sync(accel_t, accel_val, t_target_ms, WINDOW_HALF_MS) if len(accel_val) else np.full((len(t_target_ms),3), np.nan)
    synced_gyro  = window_mean_sync(gyro_t,  gyro_val,  t_target_ms, WINDOW_HALF_MS) if len(gyro_val)  else np.full((len(t_target_ms),3), np.nan)

    df_out["Ax"] = synced_accel[:,0]
    df_out["Ay"] = synced_accel[:,1]
    df_out["Az"] = synced_accel[:,2]
    df_out["Gx"] = synced_gyro[:,0]
    df_out["Gy"] = synced_gyro[:,1]
    df_out["Gz"] = synced_gyro[:,2]

    if(prnt):
        print(f"   Accel samples: {len(accel_t)}, Gyro samples: {len(gyro_t)}, Synced to {len(t_target_ms)} timestamps")

    # ============ FINAL CLEANUP & GAP FILLING  ============
    final_cols = [time_col] + gt_cols + imu_cols
    for c in final_cols:
        if c not in df_out.columns:
            df_out[c] = np.nan
    df_out = df_out[final_cols]

    # Gap filling
    t = df_out[time_col].values
    if len(t) == 0:
        raise ValueError("No samples!")

    _, uniq = np.unique(t, return_index=True)
    if len(uniq) != len(t):
        df_out = df_out.iloc[sorted(uniq)].reset_index(drop=True)
        t = df_out[time_col].values

    new_times = [t[0]]
    for i in range(len(t)-1):
        dt = t[i+1] - t[i]
        if dt > MAX_GAP_S:
            inter = np.arange(t[i] + MAX_GAP_S, t[i+1], MAX_GAP_S)
            new_times.extend(inter.tolist())
        new_times.append(t[i+1])
    new_times = np.array(sorted(set(new_times)))

    if len(new_times) == len(t) and np.allclose(new_times, t):
        df_out.to_csv(OUTPUT, index=False)
        print("Saved (no resampling needed):", OUTPUT)
    else:
        df_interp = df_out.set_index(time_col)
        df_new = pd.DataFrame(index=new_times)

        # circular yaw
        sin_y = np.sin(df_interp["ang_gt_yaw"])
        cos_y = np.cos(df_interp["ang_gt_yaw"])
        sin_i = pd.Series(sin_y, index=df_interp.index).reindex(new_times).interpolate('linear').ffill().bfill()
        cos_i = pd.Series(cos_y, index=df_interp.index).reindex(new_times).interpolate('linear').ffill().bfill()
        df_new["ang_gt_yaw"] = np.arctan2(sin_i.values, cos_i.values)

        for col in df_interp.columns:
            if col == "ang_gt_yaw": continue
            s = df_interp[col].reindex(new_times)
            df_new[col] = s.interpolate('linear').ffill().bfill().values

        df_new = df_new.reset_index().rename(columns={"index": time_col})
        for c in final_cols:
            if c not in df_new.columns:
                df_new[c] = np.nan
        df_new = df_new[final_cols]
        df_new.to_csv(OUTPUT, index=False)
        print(f"Resampled & saved: {OUTPUT} ({len(t)} → {len(new_times)} rows)")

    if(prnt):
        print("Final columns:", df_new.columns.tolist() if 'df_new' in locals() else df_out.columns.tolist())
        print(pd.read_csv(OUTPUT, nrows=8))


stage2(input_file, output_file, prnt=True)

#stage 3

Stage 3: orientation calibration (roll, pitch, yaw) from synchronized IMU+GPS.

This stage:
  - Loads the time-aligned CSV from stage2 (expects columns: time, Ax/Ay/Az, Gx/Gy/Gz).
  - Estimates device roll & pitch by low-pass filtering accelerometer to extract
    the gravity vector and fitting it to [0,0,+g].
  - Rotates accelerometer and gyroscope measurements into a leveled horizontal frame.
  - Searches for the yaw offset that minimizes lateral motion (non-holonomic cost)
    by rotating the horizontal accelerations and minimizing lateral velocity variance.
  - Optionally plots velocity/acceleration traces and the cost curve.
  - Returns (roll_deg, pitch_deg, best_yaw_deg) in degrees.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, detrend

input_file = "/content/ai-imu-dr/raw_data/data_stage_2.csv"
segment_start = 0
segment_end = None

def stage3(input_path, segment_start = 0, segment_end = None, plot = True):
    df = pd.read_csv(input_path)

    if(segment_start!=0 or segment_end!=None):
        segment_end = len(df) if segment_end==None else segment_end
        df = df[segment_start:segment_end]

    time = df['time'].values
    ax = df['Ax'].values
    ay = df['Ay'].values
    az = df['Az'].values
    gx = df['Gx'].values
    gy = df['Gy'].values
    gz = df['Gz'].values

    fs = 1 / np.mean(np.diff(time))
    print(f"Sampling frequency: {fs:.1f} Hz")

    # ============================= STEP 1: Estimate Roll & Pitch from Gravity =============================
    # Low-pass filter to get gravity vector in device frame (during all periods, including motion)
    cutoff = 0.2  # Hz
    nyq = 0.5 * fs
    b, a = butter(4, cutoff / nyq, btype='low')

    gx_f = filtfilt(b, a, ax)
    gy_f = filtfilt(b, a, ay)
    gz_f = filtfilt(b, a, az)

    # Average gravity vector over the whole sequence (or only static parts if you know them)
    g_vec = np.array([np.mean(gx_f), np.mean(gy_f), np.mean(gz_f)])
    g_norm = np.linalg.norm(g_vec)
    print(f"Average gravity vector (device frame): [{g_vec[0]:.3f}, {g_vec[1]:.3f}, {g_vec[2]:.3f}] m/s²")
    print(f"Gravity magnitude: {g_norm:.3f} m/s²")

    # Compute roll and pitch that align this gravity vector to [0, 0, +9.81]
    roll = np.arctan2(g_vec[1], np.sqrt(g_vec[0]**2 + g_vec[2]**2))
    pitch = np.arctan2(-g_vec[0], np.sqrt(g_vec[1]**2 + g_vec[2]**2))

    roll_deg = np.degrees(roll)
    pitch_deg = np.degrees(pitch)

    print(f"\n=== ESTIMATED DEVICE MOUNTING ===")
    print(f"Roll  (rotation around X):  {roll_deg:+6.3f}°")
    print(f"Pitch (rotation around Y):  {pitch_deg:+6.3f}°")

    # Build rotation matrix: R_device_to_horizontal = R_pitch @ R_roll
    cr, sr = np.cos(roll), np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)

    R_roll = np.array([[1, 0, 0],
                    [0, cr, -sr],
                    [0, sr, cr]])

    R_pitch = np.array([[cp, 0, sp],
                        [0, 1, 0],
                        [-sp, 0, cp]])

    R_dev_to_horiz = R_pitch @ R_roll  # Apply roll first, then pitch

    # ============================= STEP 2: Rotate Accel & Gyro into Horizontal Car Frame =============================
    # Rotate linear acceleration (first remove gravity in device frame, then rotate)
    lin_acc_dev = np.vstack((ax, ay, az)).T - np.vstack((gx_f, gy_f, gz_f)).T
    lin_acc_horiz = (R_dev_to_horiz @ lin_acc_dev.T).T   # Shape: (N, 3)

    # Rotate gyroscope (angular rates transform with the same rotation)
    gyro_dev = np.vstack((gx, gy, gz)).T
    gyro_horiz = (R_dev_to_horiz @ gyro_dev.T).T

    a_hx, a_hy, a_hz = lin_acc_horiz[:,0], lin_acc_horiz[:,1], lin_acc_horiz[:,2]
    g_hx, g_hy, g_hz = gyro_horiz[:,0], gyro_horiz[:,1], gyro_horiz[:,2]

    # ============================= STEP 3: Yaw Calibration in Horizontal Frame =============================
    dt = np.mean(np.diff(time))
    def yaw_cost_in_horizontal_frame(yaw_deg):
        theta = np.deg2rad(yaw_deg)
        c, s = np.cos(theta), np.sin(theta)

        # Rotate horizontal accelerations into candidate car frame
        a_forward =  c * a_hx + s * a_hy
        a_lateral = -s * a_hx + c * a_hy

        v_forward = detrend(np.cumsum(a_forward) * dt)
        v_lateral = detrend(np.cumsum(a_lateral) * dt)

        cost = np.mean(v_lateral**2)   # Main nonholonomic term
        return cost

    # Coarse search
    print("\nRunning yaw grid search in leveled horizontal frame...")
    yaws = np.arange(-180, 180, 0.8)
    costs = [yaw_cost_in_horizontal_frame(y) for y in yaws]

    best_coarse_idx = np.argmin(costs)
    best_yaw_coarse = yaws[best_coarse_idx]

    # Fine refinement
    fine_yaws = np.arange(best_yaw_coarse - 4, best_yaw_coarse + 4.1, 0.1)
    fine_costs = [yaw_cost_in_horizontal_frame(y) for y in fine_yaws]
    best_idx = np.argmin(fine_costs)
    best_yaw_deg = fine_yaws[best_idx]

    print(f"\n=== FINAL RESULT (FULL 3D CALIBRATION) ===")
    print(f"Device mounting  → Roll: {roll_deg:+6.3f}° | Pitch: {pitch_deg:+6.3f}°")
    print(f"Remaining yaw offset (horizontal → car forward): {best_yaw_deg:+6.3f}°")

    # ============================= OPTIONAL: Plot best yaw =============================
    if(plot == True):
        theta = np.deg2rad(best_yaw_deg)
        c, s = np.cos(theta), np.sin(theta)
        a_fwd = c * a_hx + s * a_hy
        a_lat = -s * a_hx + c * a_hy
        v_fwd = detrend(np.cumsum(a_fwd) * dt)
        v_lat = detrend(np.cumsum(a_lat) * dt)

        plt.figure(figsize=(12,8))
        plt.suptitle(f"Best Alignment → Roll {roll_deg:+.2f}° | Pitch {pitch_deg:+.2f}° | Yaw {best_yaw_deg:+.2f}°")

        plt.subplot(3,1,1)
        plt.plot(time - time[0], v_fwd, label='Forward velocity')
        plt.plot(time - time[0], v_lat, label='Lateral velocity')
        plt.legend(); plt.grid(); plt.ylabel('Velocity (m/s)')

        plt.subplot(3,1,2)
        plt.plot(time - time[0], a_fwd, label='a_forward')
        plt.plot(time - time[0], a_lat, label='a_lateral')
        plt.legend(); plt.grid(); plt.ylabel('Accel (m/s²)')

        plt.subplot(3,1,3)
        plt.plot(yaws, costs, 'b.-', markersize=3)
        plt.axvline(best_yaw_deg, color='r', linestyle='--')
        plt.xlabel('Yaw offset (deg)')
        plt.ylabel('Cost')
        plt.grid()
        plt.tight_layout()
        plt.show()

    return roll_deg, pitch_deg, best_yaw_deg


roll_offset, pitch_offset, yaw_offset = stage3(input_file, segment_start, segment_end, True)


#stage 4

Stage 4: rotate IMU measurements from device frame into vehicle (car) frame.

This stage:
  - Loads the time-aligned CSV (from stage2).
  - Builds a rotation from estimated mounting angles (roll, pitch, yaw).
  - Applies the rotation to accelerometer and gyroscope columns (Ax/Ay/Az, Gx/Gy/Gz).
  - Writes a new CSV with rotated IMU columns (other columns preserved).

The accuracy of results highly depends on yaw_offset. The stage 3 is still not robust in calculating yaw_offset. So in those cases other values can be tried to see results.


In [ ]:
import numpy as np
import pandas as pd

input_file = "/content/ai-imu-dr/raw_data/data_stage_2.csv"
output_file = "/content/ai-imu-dr/raw_data/data_stage_3_rotated.csv"

# default offsets from stage 3
roll = roll_offset
pitch = pitch_offset
yaw = yaw_offset

def stage4(input_path, output_path, yaw_deg, roll_deg = 0, pitch_deg = 0):
    INPUT = input_path
    OUTPUT = output_path

    roll_dg = -1* roll_deg
    pitch_dg = -1 * pitch_deg
    yaw_dg = -1 * yaw_deg


    roll_rad = np.deg2rad(roll_dg)
    pitch_rad = np.deg2rad(pitch_dg)
    yaw_rad = np.deg2rad(yaw_dg)


    imu_acc_cols = ["Ax", "Ay", "Az"]
    imu_gyr_cols = ["Gx", "Gy", "Gz"]
    # ----------------------------

    def Rx(phi):
        c = np.cos(phi); s = np.sin(phi)
        return np.array([[1.0, 0.0, 0.0],
                        [0.0,   c,  -s],
                        [0.0,   s,   c]], dtype=float)

    def Ry(theta):
        c = np.cos(theta); s = np.sin(theta)
        return np.array([[  c, 0.0,   s],
                        [0.0, 1.0,  0.0],
                        [ -s, 0.0,   c]], dtype=float)

    def Rz(psi):
        c = np.cos(psi); s = np.sin(psi)
        return np.array([[  c, -s, 0.0],
                        [  s,  c, 0.0],
                        [0.0, 0.0, 1.0]], dtype=float)

    # Build full device->car rotation matrix.
    # We remove device roll,pitch,yaw by applying inverse rotations:
    def rotation_device_to_car(roll, pitch, yaw):
        return Rz(-yaw) @ Rx(-roll) @ Ry(-pitch)

    # load CSV
    df = pd.read_csv(INPUT)

    # compute R_total from your estimated angles
    R_total = rotation_device_to_car(roll_rad, pitch_rad, yaw_rad)

    def rotate_columns(df, cols, R):
        """
        Rotate 3-vector columns (x,y,z) using rotation R: v_new = R @ v_old.
        Returns rotated Nx3 numpy array. Does not modify df in place.
        """
        arr = df[cols].to_numpy(dtype=float)   # shape (N,3)
        # apply rotation: for each row v_new = R @ v_old -> arr_rot = (R @ arr.T).T = arr.dot(R.T)
        arr_rot = arr.dot(R.T)
        return arr_rot

    # rotate IMU accel and gyro
    acc_rot = rotate_columns(df, imu_acc_cols, R_total)
    gyr_rot = rotate_columns(df, imu_gyr_cols, R_total)

    # write back
    df[imu_acc_cols] = acc_rot
    df[imu_gyr_cols] = gyr_rot

    df.to_csv(OUTPUT, index=False)
    print("Wrote rotated IMU to:", OUTPUT)

stage4(input_file, output_file, yaw, roll, pitch)

#stage 5

Stage 5: CSV -> pickle serialization for downstream model code.

This stage:
  - Loads the interpolated CSV produced earlier (stage2/stage4).
  - Validates required columns exist (time, p_gt_*, v_gt_*, ang_gt_*, Ax..Gz).
  - Builds numpy arrays for positions, velocities, attitudes and IMU controls.
  - Reorders IMU into [gx,gy,gz, ax,ay,az] per-sample and stacks into u (N,6).
  - Converts arrays to torch.float32 tensors.
  - Packs everything into a dict and pickles it to disk using Python pickle.

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle

try:
    import torch
except Exception as e:
    raise RuntimeError("This script requires PyTorch (torch). Install with `pip install torch` and re-run.") from e


input_file = "/content/ai-imu-dr/raw_data/data_stage_3_rotated.csv"
output_file = "/content/ai-imu-dr/data/2011_09_30_drive_0028_extract.p"

segment_start = 0
segment_end = None

def stage5(input_path, output_path, segment_start = 0, segment_end = None,prnt = False):

    rndm = 0
    pre_csv = input_path
    out_pickle = output_path

    # Name to store inside pickle (if None, it uses basename of pre_csv)
    name_in_pickle = None                # e.g. "2011_09_30_drive_0028_extract" or None to auto-derive

    df = pd.read_csv(pre_csv)
    if(segment_start!=0 or segment_end!=None):
        segment_end = len(df) if segment_end==None else segment_end
        df = df[segment_start:segment_end]

    required = [
        "time",
        "p_gt_x","p_gt_y","p_gt_z",
        "v_gt_x","v_gt_y","v_gt_z",
        "ang_gt_roll","ang_gt_pitch","ang_gt_yaw",
        "Ax","Ay","Az","Gx","Gy","Gz"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"pre_pickle.csv is missing required columns: {missing}")

    N = len(df)
    print(f"Rows in CSV: {N}")

    # times are assumed to be seconds relative to t0 (as in your pipeline)
    t_rel = df["time"].to_numpy(dtype=float)              # shape (N,)
    # t0_sec = float(time_offset) / 1000.0                        # convert ms -> seconds
    t0_sec = 0
    # p_gt: Nx3
    p_gt = df[["p_gt_x","p_gt_y","p_gt_z"]].to_numpy(dtype=float)

    # v_gt: Nx3
    v_gt = df[["v_gt_x","v_gt_y","v_gt_z"]].to_numpy(dtype=float)

    # ang_gt: Nx3 (roll, pitch, yaw) — ensure yaw is in radians already
    ang_gt = df[["ang_gt_roll","ang_gt_pitch","ang_gt_yaw"]].to_numpy(dtype=float)

    # u: reorder into [gx,gy,gz, ax,ay,az] if CSV is Ax,Ay,Az,Gx,Gy,Gz
    # The CSV may have columns Ax..Gz. We'll read them and place into u_order.
    ax = df["Ax"].to_numpy(dtype=float)
    ay = df["Ay"].to_numpy(dtype=float)
    az = df["Az"].to_numpy(dtype=float)
    gx = df["Gx"].to_numpy(dtype=float)
    gy = df["Gy"].to_numpy(dtype=float)
    gz = df["Gz"].to_numpy(dtype=float)

    # Final u ordering (match sample): gyro then accel
    u = np.stack([gx, gy, gz, ax, ay, az], axis=1)   # shape (N,6)

    # Convert numpy arrays to torch tensors (float32)
    ang_gt_t = torch.from_numpy(ang_gt.astype(np.float32))
    p_gt_t   = torch.from_numpy(p_gt.astype(np.float32))
    t_t      = torch.from_numpy(t_rel.astype(np.float32))
    v_gt_t   = torch.from_numpy(v_gt.astype(np.float32))
    u_t      = torch.from_numpy(u.astype(np.float32))

    # Name for pickle
    if name_in_pickle is None:
        base = os.path.splitext(os.path.basename(pre_csv))[0]
        name_in_pickle = base + "_extract"

    # Compose dict matching sample format
    out_dict = {
        "name": name_in_pickle,
        "ang_gt": ang_gt_t,
        "p_gt": p_gt_t,
        "t0": t0_sec,
        "t": t_t,
        "v_gt": v_gt_t,
        "u": u_t
    }

    # Save pickle
    with open(out_pickle, "wb") as f:
        pickle.dump(out_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    # Print summary
    if(prnt):
        print(f"Saved pickle to: {out_pickle}")
        print("Contents summary:")
        print(f" name: {out_dict['name']}")
        print(f" ang_gt: {out_dict['ang_gt'].shape}, dtype={out_dict['ang_gt'].dtype}")
        print(f" p_gt: {out_dict['p_gt'].shape}, dtype={out_dict['p_gt'].dtype}")
        print(f" t0: {out_dict['t0']}")
        print(f" t: {out_dict['t'].shape}, dtype={out_dict['t'].dtype}, first 5: {out_dict['t'][:5]}")
        print(f" v_gt: {out_dict['v_gt'].shape}, dtype={out_dict['v_gt'].dtype}")
        print(f" u: {out_dict['u'].shape}, dtype={out_dict['u'].dtype} (gyro first then accel)")

    # Quick sanity checks
    if out_dict["ang_gt"].shape[0] != N or out_dict["p_gt"].shape[0] != N:
        print("Warning: output tensor lengths do not match input CSV row count.")

stage5(input_file, output_file, segment_start, segment_end)

Following command will use generated pickle file to create proposed results.

position_xy.png in results compares ground truth path with proposed resuling path.

In [ ]:
%cd /content/ai-imu-dr/src
!python3 main_kitti.py

# Following code snippet can be used for full preprocessing of input_data.csv  and to create pickle file.

In [ ]:
'''
This preprocessing file should be used to convert neolog.csv data into pickle.
This pickle format is used by base code to calculate proposed path.
'''

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, detrend
import os
import pickle

try:
    from pyproj import Transformer # type: ignore
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False


INPUT_FILE = "/content/ai-imu-dr/raw_data/input_data.csv"
OUTPUT_FILE = "/content/ai-imu-dr/raw_data/data_stage_1.csv"

df = pd.read_csv(INPUT_FILE, sep=",", engine= "python")

cols_to_keep = [
    "GPS Time(ms)", "Lat", "Lon", "GPS Altitude", "GPS Speed", "Map Bearing",
    "Snapped Lat", "Snapped Lon",
    "Accel Sensor Time(ms)", "Ax", "Ay", "Az",
    "Gyro Sensor Time(ms)", "Gx", "Gy", "Gz",
    "Magn Sensor Time(ms)", "Mx", "My", "Mz",
    "Pressure Sensor Time(ms)", "Pressure"
]

df = df[cols_to_keep]
df = df[200:]                   #skipping some initial rows so GPS time and sensor times are closer

df.rename(columns={
    'GPS Time(ms)': 'GPST',
    'Accel Sensor Time(ms)': 'AST',
    'Gyro Sensor Time(ms)': 'GST',
    'Magn Sensor Time(ms)': 'MST',
    'Pressure Sensor Time(ms)': 'PST'
}, inplace=True)

## time is in epocs, so using time_offset as the starting time
mod = 1000
time_offset = (min(df['GPST'].iloc[0], df['AST'].iloc[0], df['GST'].iloc[0], df['MST'].iloc[0], df['PST'].iloc[0])//mod )* mod

print("Time offset : ", time_offset)
df['GPST'] = df['GPST'] - time_offset
df['AST'] = df['AST'] - time_offset
df['GST'] = df['GST'] - time_offset
df['MST'] = df['MST'] - time_offset
df['PST'] = df['PST'] - time_offset

df.to_csv(OUTPUT_FILE, index=False)
# print(f"Saved necessary columns -> {OUTPUT_FILE}")
# print(df.head(10))


#############################################################################
################################## Stage 2 ##################################
#############################################################################

# @title
# make_groundtruth_from_stage1_resample.py — FIXED IMU SYNC VERSION
# optional: pyproj gives more accurate ENU conversion

# ---------- USER PARAMETERS ----------
INPUT = "/content/ai-imu-dr/raw_data/data_stage_1.csv"   # input from your previous step
OUTPUT = "/content/ai-imu-dr/raw_data/data_stage_2.csv"  # produced file

# requested column groups / names
gt_cols = ["p_gt_x","p_gt_y","p_gt_z","v_gt_x","v_gt_y","v_gt_z",
           "ang_gt_roll","ang_gt_pitch","ang_gt_yaw"]
imu_cols = ["Ax","Ay","Az","Gx","Gy","Gz"]
time_col = "time"

# Which IMU timestamp to use as the master timebase
PREFERRED_IMU_TIME = ["AST", "GST", "GPST"]  # accel -> gyro -> gps fallback

USE_ENU = True
ROLL_PITCH_NOISE_STD = 0.0
MAX_GAP_S = 0.5

# Window for accel/gyro synchronization (ms)
WINDOW_HALF_MS = 8   # +-15 ms is safe and works perfectly on all Android phones
# Reduce to 10 or 8 only if you know your sensor batching is very tight

df = pd.read_csv(INPUT)

# Choose master timebase (prefers accel -> gyro -> gps)
imu_time = None
for cand in PREFERRED_IMU_TIME:
    if cand in df.columns:
        imu_time = cand
        break
if imu_time is None:
    raise ValueError(f"None of {PREFERRED_IMU_TIME} found. Columns: {df.columns.tolist()}")

t_target_ms = df[imu_time].values.astype(np.float64)   # master timestamps in ms

# ============ GPS -> ENU + interpolation ============
if "GPST" not in df.columns:
    df["GPST"] = df[imu_time]

gps_df = df[["GPST", "Lat", "Lon", "GPS Altitude", "GPS Speed", "Map Bearing"]].copy()
gps_df = gps_df.dropna(subset=["GPST"])
gps_df = gps_df.groupby("GPST", as_index=False).first()
gps_df = gps_df.sort_values("GPST")

t_gps = gps_df["GPST"].values.astype(np.float64)
lat_gps = gps_df["Lat"].astype(float).values
lon_gps = gps_df["Lon"].astype(float).values
alt_gps = gps_df["GPS Altitude"].astype(float).values if "GPS Altitude" in gps_df.columns else np.full_like(t_gps, np.nan)
speed_gps = gps_df["GPS Speed"].astype(float).values if "GPS Speed" in gps_df.columns else np.full_like(t_gps, np.nan)
bearing_gps = gps_df["Map Bearing"].astype(float).values if "Map Bearing" in gps_df.columns else np.full_like(t_gps, np.nan)

def latlon_to_enu(lats, lons, lat0=None, lon0=None):
    if lat0 is None: lat0 = float(np.mean(lats))
    if lon0 is None: lon0 = float(np.mean(lons))
    if HAVE_PYPROJ and USE_ENU:
        proj_str = f"+proj=aeqd +R=6378137 +lat_0={lat0} +lon_0={lon0}"
        transformer = Transformer.from_crs("epsg:4326", proj_str, always_xy=True)
        xs, ys = transformer.transform(lons.tolist(), lats.tolist())
        return np.array(xs), np.array(ys), lat0, lon0
    else:
        R = 6378137.0
        lat0_rad = np.deg2rad(lat0)
        dlat = np.deg2rad(lats - lat0)
        dlon = np.deg2rad(lons - lon0)
        x = R * dlon * np.cos(lat0_rad)
        y = R * dlat
        return x, y, lat0, lon0

if len(lat_gps) == 0:
    raise ValueError("No GPS samples found!")

x_gps, y_gps, lat0, lon0 = latlon_to_enu(lat_gps, lon_gps)

order = np.argsort(t_gps)
t_gps = t_gps[order]
x_gps = x_gps[order]; y_gps = y_gps[order]
alt_gps = alt_gps[order]; speed_gps = speed_gps[order]; bearing_gps = bearing_gps[order]

def circular_interp_deg(t_src, angles_deg, t_tgt):
    ang_rad = np.deg2rad(np.array(angles_deg, dtype=float))
    sin_i = np.interp(t_tgt, t_src, np.sin(ang_rad), left=np.nan, right=np.nan)
    cos_i = np.interp(t_tgt, t_src, np.cos(ang_rad), left=np.nan, right=np.nan)
    ang_i = np.rad2deg(np.arctan2(sin_i, cos_i))
    return (ang_i + 360.0) % 360.0

# Interpolate GPS fields
x_target    = np.interp(t_target_ms, t_gps, x_gps, left=np.nan, right=np.nan)
y_target    = np.interp(t_target_ms, t_gps, y_gps, left=np.nan, right=np.nan)
alt_target  = np.interp(t_target_ms, t_gps, alt_gps, left=np.nan, right=np.nan)
speed_target= np.interp(t_target_ms, t_gps, speed_gps, left=np.nan, right=np.nan)
bearing_target = circular_interp_deg(t_gps, bearing_gps, t_target_ms)

# Fill leading/trailing with nearest GPS
if t_target_ms[0] < t_gps[0]:
    first = np.searchsorted(t_target_ms, t_gps[0])
    for arr, val in zip([x_target,y_target,alt_target,speed_target,bearing_target],
                        [x_gps[0],y_gps[0],alt_gps[0],speed_gps[0],bearing_gps[0]]):
        arr[:first] = val
if t_target_ms[-1] > t_gps[-1]:
    last = np.searchsorted(t_target_ms, t_gps[-1], side="right")
    for arr, val in zip([x_target,y_target,alt_target,speed_target,bearing_target],
                        [x_gps[-1],y_gps[-1],alt_gps[-1],speed_gps[-1],bearing_gps[-1]]):
        arr[last:] = val

# ============ VELOCITY FROM SPEED + BEARING ============
bearing_rad = np.deg2rad(bearing_target)
vx = speed_target * np.cos(bearing_rad)
vy = speed_target * np.sin(bearing_rad)

t_s = t_target_ms / 1000.0
vz = np.zeros_like(alt_target)
if len(t_s) >= 2:
    dt = np.diff(t_s)
    dalt = np.diff(alt_target)
    valid = dt > 1e-9
    vz_mid = np.zeros_like(dalt)
    vz_mid[valid] = dalt[valid] / dt[valid]
    vz[0] = vz_mid[0]
    vz[-1] = vz_mid[-1]
    if len(vz_mid) > 1:
        vz[1:-1] = 0.5 * (vz_mid[:-1] + vz_mid[1:])

# ============ BUILD OUTPUT DATAFRAME ============
df_out = pd.DataFrame()
df_out[time_col] = t_s  # seconds

df_out["p_gt_x"] = x_target
df_out["p_gt_y"] = y_target
df_out["p_gt_z"] = alt_target
df_out["v_gt_x"] = vx
df_out["v_gt_y"] = vy
df_out["v_gt_z"] = vz

yaw_rad = (np.deg2rad(bearing_target) + np.pi) % (2*np.pi) - np.pi
df_out["ang_gt_roll"]  = 0.0
df_out["ang_gt_pitch"] = 0.0
df_out["ang_gt_yaw"]   = yaw_rad

if ROLL_PITCH_NOISE_STD > 0:
    rng = np.random.default_rng(42)
    df_out["ang_gt_roll"]  += rng.normal(0, ROLL_PITCH_NOISE_STD, len(df_out))
    df_out["ang_gt_pitch"] += rng.normal(0, ROLL_PITCH_NOISE_STD, len(df_out))

# ============ ROBUST IMU SYNCHRONIZATION (THE FIX) ============
print(f"Synchronizing IMU using ±{WINDOW_HALF_MS} ms window averaging...")

imu_parts = []
if all(c in df.columns for c in ["AST", "Ax", "Ay", "Az"]):
    accel = df[["AST", "Ax", "Ay", "Az"]].rename(columns={"AST": "t_ms"})
    accel["sensor"] = "accel"
    imu_parts.append(accel)
if all(c in df.columns for c in ["GST", "Gx", "Gy", "Gz"]):
    gyro = df[["GST", "Gx", "Gy", "Gz"]].rename(columns={"GST": "t_ms"})
    gyro["sensor"] = "gyro"
    imu_parts.append(gyro)

if not imu_parts:
    raise ValueError("No accel or gyro data found!")

imu_all = pd.concat(imu_parts, ignore_index=True)
imu_all = imu_all.sort_values("t_ms").reset_index(drop=True)

def window_mean_sync(t_native, values, t_target, half_win):
    out = np.full((len(t_target), values.shape[1]), np.nan)
    for i, t in enumerate(t_target):
        mask = (t_native >= t - half_win) & (t_native <= t + half_win)
        if mask.any():
            out[i] = np.mean(values[mask], axis=0)
        else:
            # fallback nearest
            idx = np.argmin(np.abs(t_native - t))
            out[i] = values[idx]
    return out

# Extract raw data
accel_mask = imu_all["sensor"] == "accel"
gyro_mask  = imu_all["sensor"] == "gyro"

accel_t = imu_all.loc[accel_mask, "t_ms"].values
gyro_t  = imu_all.loc[gyro_mask,  "t_ms"].values

accel_val = imu_all.loc[accel_mask, ["Ax","Ay","Az"]].values.astype(float)
gyro_val  = imu_all.loc[gyro_mask,  ["Gx","Gy","Gz"]].values.astype(float)

synced_accel = window_mean_sync(accel_t, accel_val, t_target_ms, WINDOW_HALF_MS) if len(accel_val) else np.full((len(t_target_ms),3), np.nan)
synced_gyro  = window_mean_sync(gyro_t,  gyro_val,  t_target_ms, WINDOW_HALF_MS) if len(gyro_val)  else np.full((len(t_target_ms),3), np.nan)

df_out["Ax"] = synced_accel[:,0]
df_out["Ay"] = synced_accel[:,1]
df_out["Az"] = synced_accel[:,2]
df_out["Gx"] = synced_gyro[:,0]
df_out["Gy"] = synced_gyro[:,1]
df_out["Gz"] = synced_gyro[:,2]

print(f"   Accel samples: {len(accel_t)}, Gyro samples: {len(gyro_t)}, Synced to {len(t_target_ms)} timestamps")

# ============ FINAL CLEANUP & GAP FILLING  ============
final_cols = [time_col] + gt_cols + imu_cols
for c in final_cols:
    if c not in df_out.columns:
        df_out[c] = np.nan
df_out = df_out[final_cols]

# Gap filling
t = df_out[time_col].values
if len(t) == 0:
    raise ValueError("No samples!")

_, uniq = np.unique(t, return_index=True)
if len(uniq) != len(t):
    df_out = df_out.iloc[sorted(uniq)].reset_index(drop=True)
    t = df_out[time_col].values

new_times = [t[0]]
for i in range(len(t)-1):
    dt = t[i+1] - t[i]
    if dt > MAX_GAP_S:
        inter = np.arange(t[i] + MAX_GAP_S, t[i+1], MAX_GAP_S)
        new_times.extend(inter.tolist())
    new_times.append(t[i+1])
new_times = np.array(sorted(set(new_times)))

if len(new_times) == len(t) and np.allclose(new_times, t):
    df_out.to_csv(OUTPUT, index=False)
    print("Saved (no resampling needed):", OUTPUT)
else:
    df_interp = df_out.set_index(time_col)
    df_new = pd.DataFrame(index=new_times)

    # circular yaw
    sin_y = np.sin(df_interp["ang_gt_yaw"])
    cos_y = np.cos(df_interp["ang_gt_yaw"])
    sin_i = pd.Series(sin_y, index=df_interp.index).reindex(new_times).interpolate('linear').ffill().bfill()
    cos_i = pd.Series(cos_y, index=df_interp.index).reindex(new_times).interpolate('linear').ffill().bfill()
    df_new["ang_gt_yaw"] = np.arctan2(sin_i.values, cos_i.values)

    for col in df_interp.columns:
        if col == "ang_gt_yaw": continue
        s = df_interp[col].reindex(new_times)
        df_new[col] = s.interpolate('linear').ffill().bfill().values

    df_new = df_new.reset_index().rename(columns={"index": time_col})
    for c in final_cols:
        if c not in df_new.columns:
            df_new[c] = np.nan
    df_new = df_new[final_cols]
    df_new.to_csv(OUTPUT, index=False)
    print(f"Resampled & saved: {OUTPUT} ({len(t)} → {len(new_times)} rows)")

print("Final columns:", df_new.columns.tolist() if 'df_new' in locals() else df_out.columns.tolist())
print(pd.read_csv(OUTPUT, nrows=8))


#############################################################################
################################## Stage 3 ##################################
#############################################################################


# ============================= LOAD DATA =============================
df = pd.read_csv('/content/ai-imu-dr/raw_data/data_stage_2.csv')
# df = df[22000:]
time = df['time'].values
ax = df['Ax'].values
ay = df['Ay'].values
az = df['Az'].values
gx = df['Gx'].values
gy = df['Gy'].values
gz = df['Gz'].values

fs = 1 / np.mean(np.diff(time))
print(f"Sampling frequency: {fs:.1f} Hz")

# ============================= STEP 1: Estimate Roll & Pitch from Gravity =============================
# Low-pass filter to get gravity vector in device frame (during all periods, including motion)
cutoff = 0.2  # Hz
nyq = 0.5 * fs
b, a = butter(4, cutoff / nyq, btype='low')

gx_f = filtfilt(b, a, ax)
gy_f = filtfilt(b, a, ay)
gz_f = filtfilt(b, a, az)

# Average gravity vector over the whole sequence (or only static parts if you know them)
g_vec = np.array([np.mean(gx_f), np.mean(gy_f), np.mean(gz_f)])
g_norm = np.linalg.norm(g_vec)
print(f"Average gravity vector (device frame): [{g_vec[0]:.3f}, {g_vec[1]:.3f}, {g_vec[2]:.3f}] m/s²")
print(f"Gravity magnitude: {g_norm:.3f} m/s²")

# Compute roll and pitch that align this gravity vector to [0, 0, +9.81]
roll = np.arctan2(g_vec[1], np.sqrt(g_vec[0]**2 + g_vec[2]**2))
pitch = np.arctan2(-g_vec[0], np.sqrt(g_vec[1]**2 + g_vec[2]**2))

roll_deg = np.degrees(roll)
pitch_deg = np.degrees(pitch)

print(f"\n=== ESTIMATED DEVICE MOUNTING ===")
print(f"Roll  (rotation around X):  {roll_deg:+6.3f}°")
print(f"Pitch (rotation around Y):  {pitch_deg:+6.3f}°")

# Build rotation matrix: R_device_to_horizontal = R_pitch @ R_roll
cr, sr = np.cos(roll), np.sin(roll)
cp, sp = np.cos(pitch), np.sin(pitch)

R_roll = np.array([[1, 0, 0],
                   [0, cr, -sr],
                   [0, sr, cr]])

R_pitch = np.array([[cp, 0, sp],
                    [0, 1, 0],
                    [-sp, 0, cp]])

R_dev_to_horiz = R_pitch @ R_roll  # Apply roll first, then pitch

# ============================= STEP 2: Rotate Accel & Gyro into Horizontal Car Frame =============================
# Rotate linear acceleration (first remove gravity in device frame, then rotate)
lin_acc_dev = np.vstack((ax, ay, az)).T - np.vstack((gx_f, gy_f, gz_f)).T
lin_acc_horiz = (R_dev_to_horiz @ lin_acc_dev.T).T   # Shape: (N, 3)

# Rotate gyroscope (angular rates transform with the same rotation)
gyro_dev = np.vstack((gx, gy, gz)).T
gyro_horiz = (R_dev_to_horiz @ gyro_dev.T).T

a_hx, a_hy, a_hz = lin_acc_horiz[:,0], lin_acc_horiz[:,1], lin_acc_horiz[:,2]
g_hx, g_hy, g_hz = gyro_horiz[:,0], gyro_horiz[:,1], gyro_horiz[:,2]

# ============================= STEP 3: Yaw Calibration in Horizontal Frame =============================
def yaw_cost_in_horizontal_frame(yaw_deg):
    theta = np.deg2rad(yaw_deg)
    c, s = np.cos(theta), np.sin(theta)

    # Rotate horizontal accelerations into candidate car frame
    a_forward =  c * a_hx + s * a_hy
    a_lateral = -s * a_hx + c * a_hy

    dt = np.mean(np.diff(time))
    v_forward = detrend(np.cumsum(a_forward) * dt)
    v_lateral = detrend(np.cumsum(a_lateral) * dt)

    cost = np.mean(v_lateral**2)   # Main nonholonomic term
    return cost

# Coarse search
print("\nRunning yaw grid search in leveled horizontal frame...")
yaws = np.arange(-180, 180, 0.8)
costs = [yaw_cost_in_horizontal_frame(y) for y in yaws]

best_coarse_idx = np.argmin(costs)
best_yaw_coarse = yaws[best_coarse_idx]

# Fine refinement
fine_yaws = np.arange(best_yaw_coarse - 4, best_yaw_coarse + 4.1, 0.1)
fine_costs = [yaw_cost_in_horizontal_frame(y) for y in fine_yaws]
best_idx = np.argmin(fine_costs)
best_yaw_deg = fine_yaws[best_idx]

print(f"\n=== FINAL RESULT (FULL 3D CALIBRATION) ===")
print(f"Device mounting  → Roll: {roll_deg:+6.3f}° | Pitch: {pitch_deg:+6.3f}°")
print(f"Remaining yaw offset (horizontal → car forward): {best_yaw_deg:+6.3f}°")

# ============================= OPTIONAL: Plot best yaw =============================
plot = True
if(plot == True):
    theta = np.deg2rad(best_yaw_deg)
    c, s = np.cos(theta), np.sin(theta)
    a_fwd = c * a_hx + s * a_hy
    a_lat = -s * a_hx + c * a_hy
    v_fwd = detrend(np.cumsum(a_fwd) * dt)
    v_lat = detrend(np.cumsum(a_lat) * dt)

    plt.figure(figsize=(12,8))
    plt.suptitle(f"Best Alignment → Roll {roll_deg:+.2f}° | Pitch {pitch_deg:+.2f}° | Yaw {best_yaw_deg:+.2f}°")

    plt.subplot(3,1,1)
    plt.plot(time - time[0], v_fwd, label='Forward velocity')
    plt.plot(time - time[0], v_lat, label='Lateral velocity')
    plt.legend(); plt.grid(); plt.ylabel('Velocity (m/s)')

    plt.subplot(3,1,2)
    plt.plot(time - time[0], a_fwd, label='a_forward')
    plt.plot(time - time[0], a_lat, label='a_lateral')
    plt.legend(); plt.grid(); plt.ylabel('Accel (m/s²)')

    plt.subplot(3,1,3)
    plt.plot(yaws, costs, 'b.-', markersize=3)
    plt.axvline(best_yaw_deg, color='r', linestyle='--')
    plt.xlabel('Yaw offset (deg)')
    plt.ylabel('Cost')
    plt.grid()
    plt.tight_layout()
    plt.show()



#############################################################################
################################## Stage 4 ##################################
#############################################################################


import numpy as np
import pandas as pd

# ---------- CONFIG ----------
INPUT = "/content/ai-imu-dr/raw_data/data_stage_2.csv"
OUTPUT = "/content/ai-imu-dr/raw_data/data_stage_2_rotated.csv"

# If you already estimated roll/pitch/yaw (radians), set them here.
# Example: roll,pitch from gravity-median; yaw from nonholonomic search
# roll_rad = roll_med; pitch_rad = pitch_med; yaw_rad = yaw_offset_rad
roll_dg = -1* roll_deg
pitch_dg = -1 * pitch_deg
yaw_dg = best_yaw_deg


roll_rad = (roll_dg) * 0.0174533
pitch_rad = (pitch_dg) * 0.0174533
yaw_rad = (180+best_yaw_deg) * 0.0174533


imu_acc_cols = ["Ax", "Ay", "Az"]
imu_gyr_cols = ["Gx", "Gy", "Gz"]
# ----------------------------

def Rx(phi):
    c = np.cos(phi); s = np.sin(phi)
    return np.array([[1.0, 0.0, 0.0],
                     [0.0,   c,  -s],
                     [0.0,   s,   c]], dtype=float)

def Ry(theta):
    c = np.cos(theta); s = np.sin(theta)
    return np.array([[  c, 0.0,   s],
                     [0.0, 1.0,  0.0],
                     [ -s, 0.0,   c]], dtype=float)

def Rz(psi):
    c = np.cos(psi); s = np.sin(psi)
    return np.array([[  c, -s, 0.0],
                     [  s,  c, 0.0],
                     [0.0, 0.0, 1.0]], dtype=float)

# Build full device->car rotation matrix.
# We remove device roll,pitch,yaw by applying inverse rotations:
# R_total = Rz(-yaw) @ Ry(-pitch) @ Rx(-roll)
def rotation_device_to_car(roll, pitch, yaw):
    return Rz(-yaw) @ Rx(-roll) @ Ry(-pitch)

# load CSV
df = pd.read_csv(INPUT)

# compute R_total from your estimated angles
R_total = rotation_device_to_car(roll_rad, pitch_rad, yaw_rad)

def rotate_columns(df, cols, R):
    """
    Rotate 3-vector columns (x,y,z) using rotation R: v_new = R @ v_old.
    Returns rotated Nx3 numpy array. Does not modify df in place.
    """
    assert len(cols) == 3, "Provide three columns in order [x,y,z]"
    arr = df[cols].to_numpy(dtype=float)   # shape (N,3)
    # apply rotation: for each row v_new = R @ v_old -> arr_rot = (R @ arr.T).T = arr.dot(R.T)
    arr_rot = arr.dot(R.T)
    return arr_rot

# rotate IMU accel and gyro
acc_rot = rotate_columns(df, imu_acc_cols, R_total)
gyr_rot = rotate_columns(df, imu_gyr_cols, R_total)

# write back
df[imu_acc_cols] = acc_rot
df[imu_gyr_cols] = gyr_rot

# If you want gravity-removed linear accel in car frame:
# (estimate gravity in device frame first, rotate gravity to car frame, then subtract)
# Example:
#   fs = 1.0/np.median(np.diff(df['time'].to_numpy()))
#   g_dev_x = lowpass(df['Ax'], fs, fc=0.5)
#   ... compute g_dev = [g_dev_x, g_dev_y, g_dev_z]
#   g_car = (R_total @ g_dev.T).T  # per-sample if g_dev is per-sample
#   linear_accel_car = acc_car - g_car
#
# The snippet above requires per-sample gravity estimate (low-pass) if you want to remove gravity.

df.to_csv(OUTPUT, index=False)
print("Wrote rotated IMU to:", OUTPUT)

rndm = 0

# --- Require torch for tensors ---
try:
    import torch
except Exception as e:
    raise RuntimeError("This script requires PyTorch (torch). Install with `pip install torch` and re-run.") from e

# ========== USER CONFIG ==========
# pre_csv = "/content/data_stage_smooth.csv"           # input (interpolated) CSV (created earlier)
pre_csv = "/content/ai-imu-dr/raw_data/data_stage_2_rotated.csv"           # input (interpolated) CSV (created earlier)

out_pickle = f"/content/ai-imu-dr/data/2011_09_30_drive_0028_extract.p"      # output .p file you want
# Set t0 in milliseconds (same t0 you used to create ground truth earlier)

# Name to store inside pickle (if None, it uses basename of pre_csv)
name_in_pickle = None                # e.g. "2011_09_30_drive_0028_extract" or None to auto-derive
# ==================================

# Read pre_pickle CSV
df = pd.read_csv(pre_csv)
# df = df[1500:]


# Required columns expected in pre_pickle.csv
# Must include 'time', p_gt_x/y/z, v_gt_x/y/z, ang_gt_roll/pitch/yaw, Ax Ay Az Gx Gy Gz
required = [
    "time",
    "p_gt_x","p_gt_y","p_gt_z",
    "v_gt_x","v_gt_y","v_gt_z",
    "ang_gt_roll","ang_gt_pitch","ang_gt_yaw",
    "Ax","Ay","Az","Gx","Gy","Gz"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f"pre_pickle.csv is missing required columns: {missing}")

# Number of samples
N = len(df)
print(f"Rows in CSV: {N}")

# Build arrays
# times are assumed to be seconds relative to t0 (as in your pipeline)
t_rel = df["time"].to_numpy(dtype=float)              # shape (N,)
t0_sec = float(time_offset) / 1000.0                        # convert ms -> seconds

# p_gt: Nx3
p_gt = df[["p_gt_x","p_gt_y","p_gt_z"]].to_numpy(dtype=float)

# v_gt: Nx3
v_gt = df[["v_gt_x","v_gt_y","v_gt_z"]].to_numpy(dtype=float)

# ang_gt: Nx3 (roll, pitch, yaw) — ensure yaw is in radians already
ang_gt = df[["ang_gt_roll","ang_gt_pitch","ang_gt_yaw"]].to_numpy(dtype=float)

# u: reorder into [gx,gy,gz, ax,ay,az] if CSV is Ax,Ay,Az,Gx,Gy,Gz
# The CSV may have columns Ax..Gz. We'll read them and place into u_order.
ax = df["Ax"].to_numpy(dtype=float)
ay = df["Ay"].to_numpy(dtype=float)
az = df["Az"].to_numpy(dtype=float)
gx = df["Gx"].to_numpy(dtype=float)
gy = df["Gy"].to_numpy(dtype=float)
gz = df["Gz"].to_numpy(dtype=float)

# Final u ordering (match sample): gyro then accel
u = np.stack([gx, gy, gz, ax, ay, az], axis=1)   # shape (N,6)

# Convert numpy arrays to torch tensors (float32)
ang_gt_t = torch.from_numpy(ang_gt.astype(np.float32))
p_gt_t   = torch.from_numpy(p_gt.astype(np.float32))
t_t      = torch.from_numpy(t_rel.astype(np.float32))
v_gt_t   = torch.from_numpy(v_gt.astype(np.float32))
u_t      = torch.from_numpy(u.astype(np.float32))

# Name for pickle
if name_in_pickle is None:
    base = os.path.splitext(os.path.basename(pre_csv))[0]
    # make a sensible default name
    name_in_pickle = base + "_extract"

# Compose dict matching sample format
out_dict = {
    "name": name_in_pickle,
    "ang_gt": ang_gt_t,
    "p_gt": p_gt_t,
    "t0": t0_sec,
    "t": t_t,
    "v_gt": v_gt_t,
    "u": u_t
}

# Save pickle
with open(out_pickle, "wb") as f:
    pickle.dump(out_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

# Print summary
print(f"Saved pickle to: {out_pickle}")
print("Contents summary:")
print(f" name: {out_dict['name']}")
print(f" ang_gt: {out_dict['ang_gt'].shape}, dtype={out_dict['ang_gt'].dtype}")
print(f" p_gt: {out_dict['p_gt'].shape}, dtype={out_dict['p_gt'].dtype}")
print(f" t0: {out_dict['t0']}")
print(f" t: {out_dict['t'].shape}, dtype={out_dict['t'].dtype}, first 5: {out_dict['t'][:5]}")
print(f" v_gt: {out_dict['v_gt'].shape}, dtype={out_dict['v_gt'].dtype}")
print(f" u: {out_dict['u'].shape}, dtype={out_dict['u'].dtype} (gyro first then accel)")

# Quick sanity checks
if out_dict["ang_gt"].shape[0] != N or out_dict["p_gt"].shape[0] != N:
    print("Warning: output tensor lengths do not match input CSV row count.")


In [ ]:
%cd /content/ai-imu-dr/src

!python3 main_kitti.py